# differential-expression
Per-gene tests of whether genes amplified on ecDNA are overexpressed. Produces a gene expression panel for **Fig. 2**. Full design notes: `model-design.md`.

## Requirements
- env `differential-expression.yml`
- `src/data_imports.R`, `src/gex_imports.R`
- `data/Supplementary Tables.xlsx` (S.T.2, S.T.5)
- `data/source/gene-expression/rna-metadata.tsv` (`1-collate-gex-metadata.ipynb`) and count matrices (`0-generate-gex-matrices.ipynb`)
- voom cache `data/source/gene-expression/voom-dream-weights.rds` (`gex_imports::generate_dream_cache`, about 1.5 hours)

## Methods
`variancePartition::dream`: linear mixed models on voom-weighted log-CPM. Chosen because the data have repeated measures (221 patients with libraries from more than one collection event; 38 events with more than one library) that need random effects, which DESeq2, edgeR and limma do not support (limma `duplicateCorrelation` allows only one blocking factor). voom also handles the non-integer RSEM expected counts.

**Data.**
- 2,589 RNA-seq libraries (1,566 CBTN, 1,023 SJ) linked to S.T.2 WGS biosamples by collection event (7316- sample ID for CBTN, biosample for SJ).
- Count matrices joined on gene symbol after harmonizing symbols to current official symbols (org.Hs.eg.db) and dropping ambiguous symbols; genes not in both matrices dropped (SJ: 21,494 symbols; CBTN: 22,771, including protein-coding genes absent from the SJ counts such as GAS7, CNOT2, SS18). 37,532 shared genes.
- Genes with at least 1 CPM in at least 10 libraries (30,131) kept; TMM normalization; voom weights from `voomWithDreamWeights` on all kept genes, with the covariate formula below.

**Exposure.** For each gene and WGS biosample: *ecDNA* (a full, untruncated copy on ecDNA), *chromosomal* (otherwise a full copy on a BFB, linear or complex non-cyclic amplicon), *excluded* (amplified only in unknown features, or only truncated), or *none*. Each collection event takes the highest-priority status among its WGS biosamples (ecDNA > chromosomal > excluded > none); excluded libraries are dropped. WGS and RNA come from different aliquots so for multi-region events RNA does not match DNA 1:1.

**Regression**, one gene at a time, both cohorts pooled:
`log-CPM ~ amplified + ecDNA + molecular_subtype + sex + tumor_history + cohort + rna_selection + read_length_sj + seq_center_cbtn + (1 | patient_id) + (1 | event_id)`
- `amplified` (ecDNA or chromosomal vs none) and `ecDNA` (ecDNA vs chromosomal) indicators, as in other regressions in the paper.
- `molecular_subtype`: cancer type, or cancer type x subclass where available (as in the survival analysis), fixed effect.
- `tumor_history`: Diagnosis, Relapse (progressive, recurrence, second malignancy), Autopsy, Metastasis.
- Technical covariates: `cohort` (PBTA vs SJ, also absorbs sequencing pipeline differences), `rna_selection` (poly-A vs total RNA), and cohort-specific read length (SJ) and sequencing center (CBTN).
- Random effects for multiple samples per event and per patient.
- Not adjusted for copy number or tumor fraction.

**Tests.** Three contrasts per gene, two-sided t-tests with Satterthwaite degrees of freedom: ecDNA vs none (`amplified + ecDNA`), chromosomal vs none, and ecDNA vs chromosomal. A contrast is tested only if each compared amplified group has at least `MIN_N` (3) libraries; untested contrasts are left out of the Benjamini-Hochberg correction, which is applied within each contrast across the genes shown.

## Figure
- **Diamonds, bars**: estimated log2 fold change vs unamplified tumors of the same subtype and batch, with 95% CI (t, Satterthwaite df). Not shown for groups with fewer than `MIN_N` libraries.
- **Points**: one per amplified library: observed log-CPM minus the model's prediction for an unamplified library with the same covariates (fixed effects, with `amplified` and `ecDNA` set to FALSE). Random effects are not removed, so the mean of the points differs slightly from the estimate.
- **Dashed line**: unamplified baseline.
- **Brackets**: ecDNA vs chromosomal, BH-adjusted; shown only where tested for N > 3 in both categories.
- *n*: libraries with ecDNA | chromosomal amplification of the gene.
- **C19MC** is left blank; mature miRNAs not well-captured in these data.

Small groups give wide intervals and are sensitive to single tumors (eg. MET chromosomal: one BFB amplicon at about 360 copies).

## Outputs
`out/gex_by_amplicon_class.{png,svg}`

## TODO
- Regenerate the voom cache (`generate_dream_cache`, about 1.5 hours) when the model is final: the current cache was computed with `(1 | patient_id)` only, before `(1 | event_id)` was added.

In [ ]:
Sys.setenv(LANGUAGE = "en")

suppressPackageStartupMessages({
    library(tidyverse)
    library(readxl)
    #library(edgeR)
    #library(variancePartition)
    #library(BiocParallel)
})


# imports from external file
import <- new.env()
source("../../src/data_imports.R", local = import)
source("../../src/gex_imports.R", local = import)
plotting <- new.env()
source("../../src/plotting.R",local=plotting)

sessionInfo()

In [ ]:
ST_PATH = '../../data/Supplementary Tables.xlsx'
GENES = import$genes(ST_PATH)
BIOSAMPLES = import$biosamples(ST_PATH)
VOOM_CACHE <- '../../data/source/gene-expression/voom-dream-weights.rds'


In [ ]:
rna <- import$gex_metadata() %>%
    import$prepare_gex_covariates()
if (file.exists(VOOM_CACHE)) {
    vobj <- readRDS(VOOM_CACHE)
} else {
    stop('VOOM_CACHE not found. Point to the cache or generate it using gex_imports::generate_dream_cache().')
    # imports$generate_dream_cache(rna,VOOM_CACHE) # this takes just over an hour to run.
}
stopifnot(identical(colnames(vobj), rna$rna_id))

In [ ]:
import$test_gene('MYCN',vobj,rna,BIOSAMPLES,GENES)

In [ ]:
collate_plot_data <- function(fits, min_n=3){
    results <- fits %>%
        map(import$test_gene) %>%
        list_rbind() %>%
        mutate(ecDNA_vs_none_tested = n_ecDNA >= min_n,
               intra_vs_none_tested = n_intra >= min_n,
               ecDNA_vs_intra_tested = pmin(n_ecDNA, n_intra) >= min_n,
               ecDNA_vs_none_padj = p.adjust(if_else(ecDNA_vs_none_tested, ecDNA_vs_none_p, NA), method = 'BH'),
               intra_vs_none_padj = p.adjust(if_else(intra_vs_none_tested, intra_vs_none_p, NA), method = 'BH'),
               ecDNA_vs_intra_padj = p.adjust(if_else(ecDNA_vs_intra_tested, ecDNA_vs_intra_p, NA), method = 'BH'))
    cases <- fits %>%
        map(import$gene_case_excess) %>%
        list_rbind() %>%
        filter(exposure %in% c('ecDNA','intrachromosomal'))
    return(list(results = results, cases = cases))
}

In [ ]:
plot_expression_by_amplicon_class <- function(results, cases, angle=0, min_n = 3){
    ## Fig. 2 panel: per gene, log2 expression change vs unamplified for ecDNA and chromosomal  amplifications.
    ## Points: per-case excess (gene_case_excess). Bars: model estimate with 95% CI. Brackets: ecDNA vs chromosomal, BH-adjusted.
    classes <- c(ecDNA = 'ecDNA', intrachromosomal = 'chromosomal')
    palette <- c(ecDNA = 'red', chromosomal = 'magenta')
    stars <- function(p) case_when(p < 0.001 ~ '***', p < 0.01 ~ '**', p < 0.05 ~ '*', TRUE ~ 'ns')
    genes <- results$gene
        estimates <- bind_rows(
        results %>% transmute(gene, class = 'ecDNA', n = n_ecDNA, logFC = ecDNA_vs_none_logFC, SE = ecDNA_vs_none_SE, df = ecDNA_vs_none_df),
        results %>% transmute(gene, class = 'chromosomal', n = n_intra, logFC = intra_vs_none_logFC, SE = intra_vs_none_SE, df = intra_vs_none_df)) %>%
        mutate(logFC = if_else(n >= min_n, logFC, NA),
               lo = logFC - qt(0.975, df) * SE, hi = logFC + qt(0.975, df) * SE)
    points <- cases %>% mutate(class = classes[as.character(exposure)]) %>% complete(gene = genes, class = names(palette))
    top <- bind_rows(points %>% select(gene, y = excess), estimates %>% select(gene, y = hi)) %>%
        group_by(gene) %>% summarise(y = max(y, na.rm = TRUE) + 0.8)
    brackets <- results %>% filter(!is.na(ecDNA_vs_intra_padj)) %>% transmute(gene, label = stars(ecDNA_vs_intra_padj)) %>% left_join(top, by = 'gene')
    n_labels <- results %>% transmute(gene, label = paste0('n = ', n_ecDNA, ' | n = ', n_intra))
    dodge <- position_dodge(width = 0.7)
    to_factor <- function(df) df %>% mutate(gene = factor(gene, levels = genes), across(any_of('class'), ~ factor(.x, levels = names(palette))))
    ggplot(to_factor(points), aes(x = gene, y = excess, colour = class)) +
        geom_hline(yintercept = 0, linetype = 'dashed', linewidth = 0.3, colour = 'grey50') +
        geom_point(position = position_jitterdodge(jitter.width = 0.2, dodge.width = 0.7, seed = 1), size = 1, alpha = 1, na.rm = TRUE) +
        geom_errorbar(data = to_factor(estimates), aes(y = logFC, ymin = lo, ymax = hi, group = class), position = dodge, width = .1, linewidth = 0.5, colour = 'black', na.rm = TRUE) +
        geom_point(data = to_factor(estimates), aes(y = logFC, fill = class), position = dodge, shape = 23, size = 2, colour = 'black', na.rm = TRUE) +
        geom_segment(data = to_factor(brackets), aes(x = as.numeric(gene) - 0.175, xend = as.numeric(gene) + 0.175, y = y, yend = y), inherit.aes = FALSE, linewidth = 0.3) +
        geom_text(data = to_factor(brackets), aes(x = gene, y = y, label = label), inherit.aes = FALSE, vjust = -0.3, size = 14 / .pt, family = 'Arial') +
        scale_x_discrete(labels = function(x) paste0(x, '\n', n_labels$label[match(x, n_labels$gene)])) +
        scale_colour_manual(values = palette, name = NULL) +
        scale_fill_manual(values = palette, guide = 'none') +
        labs(x = NULL, y = 'Expression vs. unamplified (log2 FC)') +
        theme_classic(base_size = 14, base_family = 'Arial') +
        theme(axis.text.x = element_text(face = 'italic', colour = 'black', angle = angle,
                                     hjust = if (angle == 0) 0.5 else 1, vjust = if (angle == 0) 0.5 else 1),
                                         legend.position = 'top', legend.key.size = unit(0.3, 'cm')) + 
        coord_cartesian(clip = 'off')
}

In [ ]:
genelist <- readLines('../genes-genomic-regions/genes.txt') %>% trimws() %>% discard(~ .x == '')
genelist <- replace(genelist, genelist == 'TTYH1', 'C19MC')
fits_expanded <- import$fit_genes(genelist, vobj, rna, BIOSAMPLES, GENES)

In [ ]:
r2 = collate_plot_data(fits_expanded)

In [ ]:
w=16;h=4;
options(repr.plot.width = w, repr.plot.height = h)

p2 <- plot_expression_by_amplicon_class(r2$results, r2$cases, angle=45)
plotting$save_ggplot('gex_by_amplicon_class_expanded', p, width = w, height = h)
p2

In [ ]:
g3 <- c("MYCN","PDGFRA","MYC","MET","CCND2","SREBF1","EGFR","CCNE1") # as in Fig. 2c
r3 <- r2$results %>% filter(gene %in% g3)
c3 <- r2$cases %>% filter(gene %in% g3)
w=6;h=4;
options(repr.plot.width = w, repr.plot.height = h)

p3 <- plot_expression_by_amplicon_class(r3, c3, angle=45)
plotting$save_ggplot('gex_by_amplicon_class_oncosubset', p, width = w, height = h)
p3

In [ ]:
r3

In [ ]:
## Archive
FIG_GENES <- c("MYCN","PDGFRA","MYC","MET","CCND2","SREBF1","EGFR","CCNE1") # as in Fig. 2c
fits <- import$fit_genes(FIG_GENES, vobj, rna, BIOSAMPLES, GENES)

r1 = collate_plot_data(fits)

w=16;h=4;
options(repr.plot.width = w, repr.plot.height = h)

p <- plot_expression_by_amplicon_class(r1$results, r1$cases)
#plotting$save_ggplot('gex_by_amplicon_class', p, width = 5, height = 2.6)
p